# YOLO11l UCF-Crime2Local — full script pipeline

Runs the same steps as [README.md](README.md), in order, from the repository root.

**Before you run:** set `REPO_ROOT` and `UCFCRIME2LOCAL_ROOT` in the next cell. The dataset folder must contain `rgb-images/`, `labels/`, `train.txt`, `test.txt`, and `labels.txt`.

**GPU:** for CUDA (e.g. Google Colab), use `TRAIN_DEVICE = "0"`. On Apple Silicon use `"mps"`, CPU use `"cpu"`.

In [ ]:
import os
from pathlib import Path

# --- edit these paths ---
REPO_ROOT = Path("/path/to/YoLoV11UCFCrime").resolve()  # this repository
UCFCRIME2LOCAL_ROOT = "/path/to/ucfcrime2local"  # UCF-Crime2Local dataset root

# Training device: "0" = first CUDA GPU, "mps", "cpu"
TRAIN_DEVICE = "0"

os.environ["UCFCRIME2LOCAL_ROOT"] = str(Path(UCFCRIME2LOCAL_ROOT).expanduser().resolve())
os.chdir(REPO_ROOT)
print("cwd:", os.getcwd())
print("UCFCRIME2LOCAL_ROOT:", os.environ["UCFCRIME2LOCAL_ROOT"])

## 0. Dependencies

Run once per environment (or skip if already installed).

In [ ]:
!pip install -q -r requirements.txt

## 1. Inspect dataset

Writes `outputs/dataset_summary.md`.

In [ ]:
!python scripts/inspect_dataset.py --out outputs/dataset_summary.md

## 2. Split videos (train / val / test)

Writes `data/splits/*.txt`. Preserves official `test.txt`; train/val from `train.txt`.

In [ ]:
!python scripts/split_videos.py --seed 42

## 3. Convert to YOLO layout

Populates `data/processed/` and `data/processed/data.yaml`.

In [ ]:
!python scripts/convert_to_yolo.py

## 4. Validate dataset + debug previews

Writes sample overlays to `outputs/debug_samples/`.

In [ ]:
!python scripts/validate_yolo_dataset.py --num-samples 20

## 5. Train YOLO11l

Default: 100 epochs, batch 16, outputs under `runs_ucfcrime/yolo11l_anomaly_region/` (or under `runs/detect/...` depending on Ultralytics layout).

If you run out of memory, lower `--batch` or use `--cache false` or `--cache disk`.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/train_yolo11l.py", "--batch", "16", "--device", TRAIN_DEVICE],
    check=True,
)

## 6. Inference on test set

Uses trained `best.pt` if present (script picks default paths).

In [ ]:
!python scripts/infer_yolo.py --weights runs_ucfcrime/yolo11l_anomaly_region/weights/best.pt

## 7. Frame anomaly scores

Writes `outputs/predictions/frame_scores_yolo.csv` (and related outputs).

In [ ]:
!python scripts/build_frame_scores.py --smooth-window 5

## 8. Detection metrics (mAP, etc.)

Runs `model.val()` and writes `outputs/metrics/metrics_detection.json`.

In [ ]:
!python scripts/evaluate_detection.py --weights runs_ucfcrime/yolo11l_anomaly_region/weights/best.pt

## 9. Anomaly metrics (optional — needs frame-level labels CSV)

Without `--frame-labels`, the script skips ROC-AUC/AP and only reminds you that scores exist.

With labels: CSV columns `frame_path` + `label`, or `video_id` + `frame_id` + `label`.

In [ ]:
# No labels (skip metrics file):
!python scripts/evaluate_anomaly.py

# Uncomment and set path when you have a labels CSV:
# !python scripts/evaluate_anomaly.py --frame-labels /path/to/frame_labels.csv

## 10. Visualization examples (optional)

In [ ]:
!python scripts/visualize_examples.py --weights runs_ucfcrime/yolo11l_anomaly_region/weights/best.pt

## 11. Merge YOLO + VadCLIP metrics (optional)

Requires `outputs/metrics/metrics_anomaly.json` from step 9 **with** frame labels. Uncomment when ready.

In [ ]:
# !python scripts/merge_metrics_optional.py --vadclip /path/to/vadclip_metrics.json